# Chronos Scramble-Trigger Experiment
### Rey Bidirectional Teleportation via Scrambling Dynamics

**Protocol:**
- **Alice_Scramble** — detects her own phase-shift events, then bursts Lorenz-energy injection for 8 s.
- **Bob_Passive** — pure passive timing capture; logs his own phase-shift event timestamps.

Run both roles simultaneously in separate Colab sessions (different Gmail accounts).
Download the JSON from each, then run `analyze_scramble_experiment.py` locally.

> **Select your role in Cell 1, then Runtime → Run all.**


In [ ]:
#@title 1 · Experiment Configuration { display-mode: "form" }
# ─────────────────────────────────────────────────────────────────────────
# ROLE:
#   Alice_Scramble  — bursts Lorenz energy at each of her own phase-shift events
#   Bob_Passive     — passive timing capture only; no injection
# ─────────────────────────────────────────────────────────────────────────
EXPERIMENT_ROLE   = 'Alice_Scramble' #@param ["Alice_Scramble", "Bob_Passive"]
DURATION_SECONDS  = 2200.0            #@param {type:"number"}

# Phase-shift detector — must be IDENTICAL on both sessions
PHASE_WINDOW      = 64    #@param {type:"number"}
PHASE_THRESHOLD   = 2.8   #@param {type:"number"}

# Compute kernel — must be IDENTICAL on both sessions
MATRIX_SIZE       = 512   #@param {type:"number"}

# Alice-only injection settings (ignored by Bob)
BURST_SCALE       = 6.0   #@param {type:"number"}
BURST_DURATION_S  = 8.0   #@param {type:"number"}

# Lorenz ODE — shared reference (identical on Alice and Bob)
LORENZ_SIGMA = 10.0    #@param {type:"number"}
LORENZ_RHO   = 28.0   #@param {type:"number"}
LORENZ_BETA  = 2.6667 #@param {type:"number"}
LORENZ_DT    = 0.01   #@param {type:"number"}

import os, time, json, pathlib
import numpy as np

# Output filename is set automatically from role — no manual rename needed
OUTPUT_FILENAME = f'/tmp/{EXPERIMENT_ROLE.lower()}_events.json'

print(f'Role:             {EXPERIMENT_ROLE}')
print(f'Duration:         {DURATION_SECONDS:.0f}s  ({DURATION_SECONDS/60:.1f} min)')
print(f'Phase window:     {PHASE_WINDOW}   threshold: {PHASE_THRESHOLD}')
print(f'Matrix size:      {MATRIX_SIZE}x{MATRIX_SIZE}')
if EXPERIMENT_ROLE == 'Alice_Scramble':
    print(f'Burst scale:      {BURST_SCALE}x   duration: {BURST_DURATION_S}s')
print(f'Output:           {OUTPUT_FILENAME}')


In [ ]:
#@title 2 · Infrastructure Fingerprint (datacenter / zone detection)
# Queries the GCP metadata server for the exact zone and instance details.
# Also does an IP geolocation lookup so we can flag if Alice and Bob
# ended up in the same datacenter (shared-resource confound).
import socket, platform, os, subprocess
import urllib.request, urllib.error, json as _json

infra = {}

# ── 1. GCP metadata server (most reliable on Colab/TPU) ──────────────────
GCP_META = 'http://metadata.google.internal/computeMetadata/v1'
GCP_HEADERS = {'Metadata-Flavor': 'Google'}

def gcp_meta(path):
    try:
        req = urllib.request.Request(f'{GCP_META}/{path}', headers=GCP_HEADERS)
        return urllib.request.urlopen(req, timeout=3).read().decode().strip()
    except Exception:
        return None

zone_full    = gcp_meta('instance/zone')          # e.g. projects/123/zones/us-central1-b
machine_type = gcp_meta('instance/machine-type')  # e.g. projects/123/machineTypes/n1-standard-4
instance_id  = gcp_meta('instance/id')
instance_name= gcp_meta('instance/name')
region       = gcp_meta('instance/region')
hostname_meta= gcp_meta('instance/hostname')
preemptible  = gcp_meta('instance/scheduling/preemptible')

# Parse short zone name from full path
zone_short = zone_full.split('/')[-1] if zone_full else None
region_short = region.split('/')[-1] if region else (zone_short[:-2] if zone_short else None)
machine_short = machine_type.split('/')[-1] if machine_type else None

infra['gcp_zone']         = zone_short
infra['gcp_region']       = region_short
infra['gcp_machine_type'] = machine_short
infra['gcp_instance_id']  = instance_id
infra['gcp_instance_name']= instance_name
infra['gcp_preemptible']  = preemptible
infra['gcp_hostname']     = hostname_meta

# ── 2. External IP + geolocation ─────────────────────────────────────────
try:
    raw = urllib.request.urlopen('https://ipinfo.io/json', timeout=5).read()
    geo = _json.loads(raw)
    infra['external_ip']  = geo.get('ip')
    infra['geo_city']     = geo.get('city')
    infra['geo_region']   = geo.get('region')
    infra['geo_country']  = geo.get('country')
    infra['geo_org']      = geo.get('org')       # e.g. 'AS15169 Google LLC'
    infra['geo_timezone'] = geo.get('timezone')  # e.g. 'America/Chicago'
    infra['geo_loc']      = geo.get('loc')       # 'lat,lon'
except Exception as e:
    infra['geo_error'] = str(e)

# ── 3. Host-level identifiers ────────────────────────────────────────────
infra['hostname']      = socket.gethostname()
infra['platform']      = platform.platform()
infra['python']        = platform.python_version()
infra['cpu_count']     = os.cpu_count()
infra['env_zone']      = os.environ.get('ZONE', os.environ.get('GCP_ZONE'))

# ── 4. TPU / accelerator info ────────────────────────────────────────────
try:
    import jax
    devices = jax.devices()
    infra['jax_devices'] = [str(d) for d in devices]
    infra['jax_backend'] = jax.default_backend()
except Exception:
    pass

try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,uuid',
                             '--format=csv,noheader'],
                            capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        infra['gpu_info'] = result.stdout.strip()
except Exception:
    pass

# ── Print summary ─────────────────────────────────────────────────────────
print('=' * 55)
print('INFRASTRUCTURE FINGERPRINT')
print('=' * 55)
print(f"  GCP zone:        {infra.get('gcp_zone', 'N/A')}")
print(f"  GCP region:      {infra.get('gcp_region', 'N/A')}")
print(f"  Machine type:    {infra.get('gcp_machine_type', 'N/A')}")
print(f"  Instance ID:     {infra.get('gcp_instance_id', 'N/A')}")
print(f"  Preemptible:     {infra.get('gcp_preemptible', 'N/A')}")
print(f"  External IP:     {infra.get('external_ip', 'N/A')}")
print(f"  Geo location:    {infra.get('geo_city')}, {infra.get('geo_region')}, {infra.get('geo_country')}")
print(f"  Timezone:        {infra.get('geo_timezone', 'N/A')}")
print(f"  Lat/Lon:         {infra.get('geo_loc', 'N/A')}")
print(f"  Org/ASN:         {infra.get('geo_org', 'N/A')}")
print(f"  Hostname:        {infra.get('hostname', 'N/A')}")
print(f"  CPU cores:       {infra.get('cpu_count', 'N/A')}")
if 'jax_backend' in infra:
    print(f"  JAX backend:     {infra['jax_backend']}")
    print(f"  JAX devices:     {infra['jax_devices']}")
print('=' * 55)
print()
print('>>> SHARE THIS BLOCK with the other session to confirm')
print('>>> different zones before starting the capture loop.')


In [ ]:
#@title 3 · Pre-compute Lorenz trajectory (deterministic, shared seed)
# Both roles pre-compute the same trajectory so the reference path can be
# reconstructed during offline analysis without needing Alice's event log.

N_LORENZ = int(DURATION_SECONDS / LORENZ_DT) + 2000

lorenz_xyz = np.zeros((N_LORENZ, 3), dtype=np.float32)
x, y, z = 0.1, 0.0, 0.0
for i in range(N_LORENZ):
    dx = LORENZ_SIGMA * (y - x)
    dy = x * (LORENZ_RHO - z) - y
    dz = x * y - LORENZ_BETA * z
    x += dx * LORENZ_DT
    y += dy * LORENZ_DT
    z += dz * LORENZ_DT
    lorenz_xyz[i] = [x, y, z]

lorenz_A = np.sqrt((lorenz_xyz ** 2).sum(axis=1)) / 10.0
print(f'Lorenz pre-computed: {N_LORENZ} steps')
print(f'A(t) range: [{lorenz_A.min():.3f}, {lorenz_A.max():.3f}]  mean={lorenz_A.mean():.3f}')


In [ ]:
#@title 4 · Phase-shift detector and timing kernel

def is_phase_shift(window):
    """True when rolling timing variance/mean ratio exceeds threshold."""
    if len(window) < PHASE_WINDOW:
        return False
    w = np.array(window[-PHASE_WINDOW:], dtype=np.float64)
    return (np.std(w) / (np.mean(np.abs(w)) + 1e-9)) > PHASE_THRESHOLD

# Fixed right-hand matrix — identical seed on both sessions
B_REF = np.random.default_rng(seed=42).standard_normal(
    (MATRIX_SIZE, MATRIX_SIZE)).astype(np.float32)

def timed_matmul(scale=1.0, lorenz_a=0.0):
    A = (np.random.randn(MATRIX_SIZE, MATRIX_SIZE).astype(np.float32)
         * scale * (1.0 + lorenz_a))
    t1 = time.perf_counter()
    _ = A @ B_REF
    return (time.perf_counter() - t1) * 1000.0

# Warm-up
_ = timed_matmul()
print(f'Warm-up matmul: {timed_matmul():.2f} ms')
print('Detector ready.')


In [ ]:
#@title 5 · Capture loop — blocks for ~36 min

events     = []
window     = []
lorenz_idx = 0
inj_scale  = 1.0
in_burst   = False
burst_end  = 0.0
n_packets  = 0
t0         = time.time()

print(f'[{EXPERIMENT_ROLE}] Starting at {time.strftime("%H:%M:%S")}')
finish_at = time.strftime('%H:%M:%S', time.localtime(t0 + DURATION_SECONDS))
print(f'Expected finish:  {finish_at}')
print()

while time.time() - t0 < DURATION_SECONDS:
    now        = time.time()
    lorenz_idx = (lorenz_idx + 1) % N_LORENZ
    A_val      = float(lorenz_A[lorenz_idx])

    # Alice: exit burst when timer expires
    if EXPERIMENT_ROLE == 'Alice_Scramble' and in_burst and now >= burst_end:
        in_burst  = False
        inj_scale = 1.0
        window.clear()   # reset detector after burst settles

    # Timed compute packet
    use_lorenz = A_val if EXPERIMENT_ROLE == 'Alice_Scramble' else 0.0
    dt_ms = timed_matmul(scale=inj_scale, lorenz_a=use_lorenz)
    window.append(dt_ms)
    n_packets += 1

    # Detect phase shift
    if not in_burst and is_phase_shift(window):
        event_t = time.time()
        elapsed = event_t - t0
        events.append({
            't':         event_t,
            'elapsed':   elapsed,
            'lorenz_A':  A_val,
            'dt_ms':     dt_ms,
            'n_packet':  n_packets,
        })

        if EXPERIMENT_ROLE == 'Alice_Scramble':
            # INJECT: Lorenz burst at scrambling moment
            inj_scale = BURST_SCALE * (1.0 + A_val)
            burst_end = event_t + BURST_DURATION_S
            in_burst  = True

        if len(events) % 5 == 0:
            tag = 'BURST' if EXPERIMENT_ROLE == 'Alice_Scramble' else 'EVENT'
            print(f'  [{elapsed:7.1f}s] {tag} #{len(events):3d}  '
                  f'A={A_val:.3f}  dt={dt_ms:.2f}ms')

    time.sleep(0.005)

elapsed_total = time.time() - t0
print()
print(f'Done. {elapsed_total:.1f}s   Packets: {n_packets}   Events: {len(events)}')
if len(events) >= 2:
    gaps = np.diff([e['elapsed'] for e in events])
    print(f'Inter-event mean: {gaps.mean():.1f}s  std={gaps.std():.1f}s')


In [ ]:
#@title 6 · Save output and download
# File is named automatically:
#   Alice_Scramble  →  /tmp/alice_scramble_events.json
#   Bob_Passive     →  /tmp/bob_passive_events.json

output = {
    'role':      EXPERIMENT_ROLE,
    't0':        t0,
    'duration':  elapsed_total,
    'n_packets': n_packets,
    'events':    events,
    'infrastructure': infra,
    'config': {
        'matrix_size':      MATRIX_SIZE,
        'burst_scale':      BURST_SCALE       if EXPERIMENT_ROLE == 'Alice_Scramble' else None,
        'burst_duration_s': BURST_DURATION_S  if EXPERIMENT_ROLE == 'Alice_Scramble' else None,
        'phase_window':     PHASE_WINDOW,
        'phase_threshold':  PHASE_THRESHOLD,
        'lorenz_sigma':     LORENZ_SIGMA,
        'lorenz_rho':       LORENZ_RHO,
        'lorenz_beta':      LORENZ_BETA,
        'lorenz_dt':        LORENZ_DT,
    },
}

with open(OUTPUT_FILENAME, 'w') as f:
    json.dump(output, f, indent=2)
print(f'Saved: {OUTPUT_FILENAME}')

try:
    from google.colab import files
    files.download(OUTPUT_FILENAME)
    print('Download triggered — check your browser downloads folder.')
except Exception:
    print(f'Not in Colab — file is at: {OUTPUT_FILENAME}')


## After both sessions finish

Move both downloaded JSON files to your project directory, then run locally:

```powershell
python python/analyze_scramble_experiment.py `
    --alice alice_scramble_events.json `
    --bob   bob_passive_events.json
```

**What to look for:**
- `Observed / Chance` ratio >> 1.0 at any lag offset
- `p-value < 0.05` in the 10,000-iteration permutation test
- Peak lag near **0 s** = simultaneous coupling
- Peak lag **positive** = Bob responds after Alice (causal)
- Peak lag **negative** = out-of-time order (Rey's OTOC prediction)
